# การบ้านสัปดาห์ที่ 6 — Titanic End-to-End

### วิชา CE564 เครื่องจักรกลปัญญาเสมือน · อ.วสุ ทาแก้ว

การบ้านชิ้นนี้มัดรวมสามสัปดาห์ไว้ในงานเดียว
คือ data preparation (สัปดาห์ที่ 3) กับ machine learning (สัปดาห์ที่ 1 ถึง 4)
และ deep learning (สัปดาห์ที่ 5 ถึง 6)

**โจทย์** ทำนายว่าผู้โดยสาร Titanic คนไหนรอดชีวิต โดยสร้างสองโมเดลบนข้อมูลชุดเดียวกัน
แล้วตัดสินด้วยหลักฐานว่าควรใช้ตัวไหน

| | |
|---|---|
| คะแนนเต็ม | 20 คะแนน (+2 คะแนนพิเศษ) |
| ไฟล์ที่ส่ง | `CE564_W06_Assignment_Titanic.ipynb` ที่รันครบมี output ทุกเซลล์ |
| กำหนดส่ง | *(อาจารย์ประกาศในคาบ)* |

**เกณฑ์ให้คะแนน**

| ส่วน | สิ่งที่วัด | คะแนน |
|---|---|---|
| 1 | เตรียมข้อมูล ทำความสะอาดครบ ไม่เหลือ NaN และไม่เหลือ object | 4 |
| 2 | แบ่งข้อมูลสามกอง และทำ scaling โดยไม่เกิด data leakage | 3 |
| 3 | Baseline ด้วย Random Forest หรือ XGBoost | 3 |
| 4 | Dense NN เขียน `nn.Module` และ training loop 5 ขั้นด้วยตัวเอง | 5 |
| 5 | ตารางและกราฟเทียบผลสองโมเดล | 2 |
| 6 | สรุปเชิงวิจารณญาณ ตอบคำถาม 4 ข้อ | 3 |
| พิเศษ | ส่งผลขึ้น Kaggle Titanic leaderboard แล้วแนบภาพคะแนน | +2 |

> **นโยบายการใช้ AI** ใช้ได้ แต่ต้องเข้าใจและอธิบายผลงานตัวเองได้ทั้งหมด
> และต้องกรอกตารางเปิดเผยการใช้ AI ท้ายไฟล์ อาจารย์อาจสุ่มสัมภาษณ์ให้อธิบายโค้ดที่ส่งมา
> การใช้ AI โดยไม่เปิดเผย หรืออธิบายงานตัวเองไม่ได้ ถือว่าผิดระเบียบความซื่อสัตย์ทางวิชาการ

---

## ส่วนที่ 0 — เตรียมเครื่องมือ (ให้มาแล้ว ไม่ต้องแก้)

In [ ]:
# ===== เซลล์บังคับ ต้องรันก่อน import torch หรือ xgboost เสมอ =====
import os
import platform

if platform.system() == 'Darwin':
    os.environ.setdefault('OMP_NUM_THREADS', '1')

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device =', DEVICE)

---

## ส่วนที่ 1 — เตรียมข้อมูล (4 คะแนน)

เซลล์โหลดข้อมูลให้มาแล้ว หน้าที่ของคุณคือทำความสะอาดต่อ

**สิ่งที่ต้องทำให้ครบ**

1. เลือก feature ที่จะใช้ และอธิบายเหตุผลว่าทำไมตัดบางคอลัมน์ทิ้ง
2. เติมค่าที่หายไปให้หมด (`age`, `embarked`, `fare` มักมีค่าหาย)
3. แปลงคอลัมน์ข้อความเป็นตัวเลข เลือกวิธีให้ตรงชนิดของหมวดหมู่
4. ตรวจให้แน่ใจว่าไม่เหลือ NaN และไม่เหลือคอลัมน์ชนิด object

*คำใบ้* ย้อนดู `CE564_W03_Workshop_DataPrep.ipynb` ได้เลย ใช้แนวทางเดียวกัน
คอลัมน์อย่าง `deck` มีค่าหายเกินครึ่ง ส่วน `alive` คือคำตอบที่รั่วมาตรงๆ ต้องทิ้ง

In [ ]:
# ===== โหลดข้อมูล (ให้มาแล้ว) =====
try:
    df = sns.load_dataset('titanic')
except Exception:  # noqa: BLE001
    from sklearn.datasets import fetch_openml
    raw = fetch_openml('titanic', version=1, as_frame=True).frame
    df = pd.DataFrame({
        'survived': raw['survived'].astype(int),
        'pclass': raw['pclass'].astype(int),
        'sex': raw['sex'],
        'age': pd.to_numeric(raw['age'], errors='coerce'),
        'sibsp': pd.to_numeric(raw['sibsp'], errors='coerce'),
        'parch': pd.to_numeric(raw['parch'], errors='coerce'),
        'fare': pd.to_numeric(raw['fare'], errors='coerce'),
        'embarked': raw['embarked'],
    })

print('ขนาดข้อมูล:', df.shape)
print('\nค่าที่หายไปในแต่ละคอลัมน์')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

In [ ]:
# ===== ส่วนที่ 1: ทำความสะอาดข้อมูล =====
# TODO 1.1 เลือกคอลัมน์ที่จะใช้ (ระวังคอลัมน์ที่รั่วคำตอบ เช่น alive)
FEATURES = []          # <<< เติมชื่อคอลัมน์ที่จะใช้
TARGET = 'survived'

# TODO 1.2 เติมค่าที่หายไป


# TODO 1.3 แปลงข้อความเป็นตัวเลข


# TODO 1.4 ตรวจผล ต้องได้ 0 ทั้งสองบรรทัด
# print('NaN ที่เหลือ:', data.isnull().sum().sum())
# print('คอลัมน์ object ที่เหลือ:', (data.dtypes == 'object').sum())

---

## ส่วนที่ 2 — แบ่งข้อมูลและปรับสเกล (3 คะแนน)

**สิ่งที่ต้องทำ**

1. แบ่งข้อมูลเป็นสามกอง train, validation, test สัดส่วนประมาณ 60/20/20
2. ใช้ `stratify` เพราะสัดส่วนคนรอดกับไม่รอดไม่เท่ากัน
3. ทำ `StandardScaler` โดย **fit บน train เท่านั้น** แล้ว transform ทั้งสามกอง
4. แปลงเป็น tensor ให้พร้อมใช้กับ PyTorch

> **ระวัง** ถ้า fit scaler บนข้อมูลทั้งก้อนก่อนแบ่ง คือ data leakage ซึ่งหักคะแนน
> เพราะข้อมูล test รั่วเข้ามาบอกค่าเฉลี่ยกับส่วนเบี่ยงเบนให้ตอนเทรน

In [ ]:
# ===== ส่วนที่ 2: แบ่งข้อมูลและปรับสเกล =====
# TODO 2.1 แยก X กับ y ออกจาก data


# TODO 2.2 แบ่งสามกอง 60/20/20 ใช้ stratify และ random_state=SEED


# TODO 2.3 fit scaler บน train เท่านั้น แล้ว transform ทั้งสามกอง


# TODO 2.4 แปลงเป็น tensor ย้ายขึ้น DEVICE
#   binary classification ต้องการ y เป็น float shape (N, 1)

---

## ส่วนที่ 3 — Baseline ด้วย tree (3 คะแนน)

ก่อนจะสร้างของใหม่ ต้องมีตัวเทียบก่อนเสมอ ถ้าไม่มี baseline เราจะไม่รู้เลยว่าโมเดลใหม่ดีจริงไหม

**สิ่งที่ต้องทำ**

1. เทรน `RandomForestClassifier` หรือ `XGBClassifier` บนข้อมูล train
2. วัด accuracy ทั้งบน val และ test
3. จับเวลาที่ใช้เทรน

> **สังเกต** tree ไม่ต้องใช้ข้อมูลที่ scale แล้ว ใส่ `X_train` ดิบเข้าไปได้เลย
> จุดนี้เองคือข้อได้เปรียบข้อหนึ่งของ tree ที่จะเอาไปเขียนในสรุปส่วนที่ 6

In [ ]:
# ===== ส่วนที่ 3: baseline =====
# TODO 3.1 เทรน RandomForest หรือ XGBoost (ใช้ X_train ที่ยังไม่ scale)


# TODO 3.2 วัด accuracy บน val และ test พร้อมจับเวลา

---

## ส่วนที่ 4 — Dense NN (5 คะแนน)

ส่วนที่ให้คะแนนมากที่สุด และเป็นหัวใจของการบ้านชิ้นนี้

**สิ่งที่ต้องทำ**

1. เขียน `nn.Module` ด้วยตัวเอง มีชั้นซ่อนอย่างน้อย 1 ชั้น
2. เลือก output layer กับ loss ให้ถูกกับโจทย์ binary
3. **เขียน training loop 5 ขั้นเอง ห้ามใช้ helper หรือ library ที่ห่อ loop ไว้**
4. เก็บ train loss กับ val loss ทุก epoch แล้วพล็อตออกมา
5. ใช้ validation เลือกว่าจะหยุดตอนไหน แล้วค่อยวัด test ครั้งเดียว

> **ห้ามแตะ test set ระหว่างเลือกโมเดล** ใช้ val ตัดสินใจทุกอย่าง
> แล้วค่อยเปิด test ตอนท้ายสุดรอบเดียว ข้อนี้มีคะแนนแยก

In [ ]:
# ===== ส่วนที่ 4: Dense NN =====
# TODO 4.1 เขียนคลาสโมเดล
class TitanicNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # TODO ประกาศชั้นที่ต้องใช้
        pass

    def forward(self, x):
        # TODO บอกว่าข้อมูลไหลยังไง
        pass


# TODO 4.2 สร้างโมเดล เลือก loss เลือก optimizer


# TODO 4.3 เขียน loop 5 ขั้น เก็บ train loss กับ val loss ทุก epoch
#   อย่าลืม model.train() กับ model.eval() ถ้าใช้ dropout


# TODO 4.4 พล็อต train loss เทียบ val loss


# TODO 4.5 โหลดน้ำหนักตอน val ดีที่สุดกลับมา แล้ววัด test ครั้งเดียว

---

## ส่วนที่ 5 — เทียบผล (2 คะแนน)

สร้างตารางและกราฟเทียบสองโมเดลบนแกนเดียวกัน ต้องมีอย่างน้อย accuracy กับเวลาที่ใช้เทรน

In [ ]:
# ===== ส่วนที่ 5: เทียบผล =====
# TODO 5.1 ทำตารางเทียบ (ใช้ pd.DataFrame ได้)


# TODO 5.2 พล็อตกราฟแท่งเทียบ accuracy และเวลา

---

## ส่วนที่ 6 — สรุปเชิงวิจารณญาณ (3 คะแนน)

ตอบสี่ข้อนี้เป็นภาษาไทย ข้อละ 3 ถึง 5 บรรทัด **ตอบจากตัวเลขที่ตัวเองรันได้จริง**
ไม่ใช่ตอบจากที่อาจารย์พูดในคาบ

**6.1** โมเดลไหนได้ accuracy สูงกว่าบน test set และห่างกันกี่ตัวอย่าง
(ลองคูณเปอร์เซ็นต์ที่ต่างกันด้วยจำนวนแถวใน test set ดู)

*ตอบ:*

**6.2** ถ้าจะส่งขึ้นใช้งานจริง คุณจะเลือกโมเดลไหน เพราะอะไร
ให้พิจารณาทั้งความแม่น เวลาที่ใช้ ความยากในการดูแล และความสามารถในการอธิบายผลให้คนอื่นฟัง

*ตอบ:*

**6.3** ข้อมูล Titanic มีคนไม่รอดมากกว่าคนรอด ถ้าโมเดลทายว่า "ไม่รอด" ทุกคนจะได้ accuracy เท่าไร
แล้วโมเดลของคุณดีกว่าการทายมั่วแบบนั้นแค่ไหน

*ตอบ:*

**6.4** จากที่ทำมาทั้งชิ้น คุณคิดว่าเมื่อไรควรใช้ neural network และเมื่อไรควรใช้ tree

*ตอบ:*

---

## ส่วนพิเศษ — Kaggle Titanic (+2 คะแนน)

ไม่บังคับ ทำแล้วได้คะแนนเพิ่ม

1. สมัคร Kaggle แล้วเข้าการแข่งขัน [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)
2. โหลด `train.csv` กับ `test.csv` จากหน้านั้น (คนละไฟล์กับที่ใช้ในคาบ คอลัมน์ไม่เหมือนกันเป๊ะ)
3. ทำ pipeline เดียวกับข้างบน แล้วสร้างไฟล์ `submission.csv` ที่มีสองคอลัมน์คือ `PassengerId` และ `Survived`
4. ส่งขึ้น leaderboard แล้วแนบภาพหน้าจอคะแนนที่ได้ลงในเซลล์ markdown ข้างล่าง

*แนบภาพและเขียนคะแนนที่ได้ตรงนี้:*

---

## ตารางเปิดเผยการใช้ AI (บังคับกรอก)

ตามนโยบายของรายวิชา ต้องกรอกตารางนี้ให้ครบทุกช่อง ถ้าไม่ได้ใช้ AI เลยให้เขียนว่า "ไม่ได้ใช้"

| หัวข้อ | คำตอบ |
|---|---|
| ใช้ AI ส่วนไหนบ้าง | |
| เครื่องมืออะไร | |
| prompt หรือแนวทางที่ใช้ | |
| ตรวจสอบและปรับปรุงผลลัพธ์อย่างไร | |

---

## เช็กลิสต์ก่อนส่ง

- [ ] รันไฟล์ใหม่ทั้งไฟล์ตั้งแต่ต้น (Kernel แล้ว Restart and Run All) ไม่มี error
- [ ] ทุกเซลล์มี output แสดงอยู่
- [ ] ไม่มีคอลัมน์ที่รั่วคำตอบอยู่ใน `FEATURES`
- [ ] `fit` scaler บน train เท่านั้น
- [ ] เขียน training loop 5 ขั้นเอง ไม่ได้ใช้ helper ห่อไว้
- [ ] ตอบคำถามส่วนที่ 6 ครบทั้งสี่ข้อ
- [ ] กรอกตารางเปิดเผยการใช้ AI แล้ว
- [ ] เปลี่ยนชื่อไฟล์เป็น `CE564_W06_Assignment_Titanic_รหัสนักศึกษา.ipynb`